# Solutions · Chapter 01-03 · NumPy

Worked answers with reasoning. E5 and E14 are the two worth slowing down for: one is about
broadcasting succeeding when you did not want it to, the other about a clever trick that is the
wrong answer at scale.

Self-contained: run from the top with a fresh kernel.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

rng = np.random.default_rng(5)
n_rows = 300
X = np.column_stack([rng.normal(20, 5, n_rows), rng.normal(60, 20, n_rows),
                     rng.normal(1000, 300, n_rows)])
y = 2.0 * X[:, 0] - 0.5 * X[:, 1] + 0.01 * X[:, 2] + rng.normal(0, 3, n_rows)
print("X", X.shape, " y", y.shape)

## E1 · The axis rule

> `axis=n` is the axis that disappears.

An array of shape `(50, 4)` summed with `axis=1` loses the 4 and returns shape **`(50,)`** - one
total per row.

If rows are observations, that is "one number per observation", which is occasionally what you
want (a row total) and usually a sign you meant `axis=0`.

## E2 · `arr * 2` versus `list * 2`

For a NumPy array, `*` is **arithmetic applied to every element**, because an array is a block of
numbers of one type and elementwise arithmetic is the whole point of it.

For a Python list, `*` is **repetition** - a list is a general-purpose container that might hold
strings or dicts, so "multiply the contents" has no meaning, and Python long ago assigned `*` to
concatenation instead.

**The practical consequence:** `list + list` concatenates while `array + array` adds elementwise,
and both run. If a total is suspiciously twice as long as expected, you are holding a list when
you thought you had an array. `np.asarray(x)` at the top of a function removes the ambiguity.

## E3 · View or copy?

A **view** is a window onto the same memory - writing through it changes the original. A **copy**
is independent.

`arr[:5]` is a **view**. So is any plain slice, and so is `.reshape(...)` when it can be done
without moving data.

`arr[[0, 1, 2]]` (fancy indexing) and `arr[mask]` (boolean indexing) always return **copies**,
because the elements they select are not necessarily contiguous in memory.

That inconsistency is not something you can derive - you have to know it. The safe rule: if you
are about to modify an array that came from another array, call `.copy()` unless you have
deliberately decided you want the write to propagate.

## E4 · By hand

In [ ]:
M = np.array([[1, 2], [3, 4], [5, 6]])
print("M.shape           ", M.shape)
print("M.sum(axis=0)     ", M.sum(axis=0), " shape", M.sum(axis=0).shape)
print("M.sum(axis=1)     ", M.sum(axis=1), " shape", M.sum(axis=1).shape)
print("M.sum()           ", M.sum())
print("(M - M.mean(0)).shape", (M - M.mean(axis=0)).shape)

| | Answer | Reasoning |
|---|---|---|
| (a) `M.shape` | `(3, 2)` | Three rows, two columns |
| (b) `M.sum(axis=0)` | `[9, 12]` | The 3 disappears: column totals `1+3+5` and `2+4+6` |
| (c) `M.sum(axis=1)` | `[3, 7, 11]` | The 2 disappears: row totals |
| (d) `M.sum()` | `21` | Everything collapses to a scalar |
| (e) shape of `M - M.mean(axis=0)` | `(3, 2)` | `(2,)` broadcasts across the rows; the shape is unchanged |

(e) is the one to notice. Subtracting the column means gives back the **same shape**, which is why
a wrong axis here does not announce itself - you asked for a `(3, 2)` array and you got one.

## E5 · Which broadcasts?

In [ ]:
for shape_a, shape_b in [((4, 3), (3,)), ((4, 3), (4,)), ((4, 3), (4, 1)),
                         ((4, 1), (1, 3)), ((4, 3), (1, 3))]:
    try:
        result = (np.ones(shape_a) + np.ones(shape_b)).shape
        print(f"{str(shape_a):<8} + {str(shape_b):<8} -> {result}")
    except ValueError as exc:
        print(f"{str(shape_a):<8} + {str(shape_b):<8} -> ERROR: {str(exc)[:52]}")

| Pair | Result | Why |
|---|---|---|
| (a) `(4,3)` + `(3,)` | `(4, 3)` | Aligned right: 3 matches 3, the `(3,)` is stretched down 4 rows |
| (b) `(4,3)` + `(4,)` | **error** | Aligned right: 3 against 4. No match, no size-1 |
| (c) `(4,3)` + `(4,1)` | `(4, 3)` | 3 against 1 -> stretch; 4 against 4 -> match |
| (d) `(4,1)` + `(1,3)` | `(4, 3)` | **Both** get stretched |
| (e) `(4,3)` + `(1,3)` | `(4, 3)` | The single row is repeated |

**(d) is the dangerous one.** Two small arrays - one with 4 elements, one with 3 - combine into a
12-element array, silently. Nobody asks for that on purpose. It happens when a column vector meets
a row vector, typically because something that should have been flattened was not, or because a
`keepdims=True` was left in.

The tell is memory and time: an operation that should have been instant takes seconds, or an array
you expected to have `n` elements has `n²`. **If an array is unexpectedly large, suspect a
broadcast.**

(b) failing is the good case. An error you see beats a wrong answer you do not.

## E6 · `standardise` with its own checks

In [ ]:
def standardise(features):
    """Centre and scale each COLUMN. Rows are observations, columns are features."""
    z = (features - features.mean(axis=0)) / features.std(axis=0)
    assert np.allclose(z.mean(axis=0), 0), "columns are not centred - wrong axis?"
    assert np.allclose(z.std(axis=0), 1), "columns are not unit scale - wrong axis?"
    return z


Z = standardise(X)
print("ok:", Z.shape, "column means", Z.mean(axis=0).round(9), "column sds", Z.std(axis=0).round(9))

try:
    bad = (X - X.mean(axis=1, keepdims=True)) / X.std(axis=1, keepdims=True)
    assert np.allclose(bad.mean(axis=0), 0), "columns are not centred - wrong axis?"
except AssertionError as exc:
    print("the buggy version is caught:", exc)

Two asserts, and the bug that cost 5.75 MAE in the failure lab now stops the notebook with a
message that names the cause.

**The principle is bigger than this function.** An assert is a claim about what your code just
did, written at the moment you still remember what you meant. It costs one line and it fires at
the point of the mistake rather than three steps downstream where the symptom appears.

**The limitation, stated honestly:** this function is still wrong for machine learning, because
it computes the mean and standard deviation from *all* the rows. In a real pipeline those must
come from the training rows only, and be applied unchanged to the test rows - otherwise the test
set has influenced the transformation, which is leakage. `StandardScaler` inside a `Pipeline`
enforces that separation; 04-06 and 04-07 build it.

## E7 · MAE and the worst prediction, in NumPy

In [ ]:
model = LinearRegression().fit(Z[:200], y[:200])
y_true, y_pred = y[200:], model.predict(Z[200:])

mae = np.abs(y_true - y_pred).mean()
worst = np.argmax(np.abs(y_true - y_pred))

print(f"MAE {mae:.3f}")
print(f"worst prediction at position {worst}: actual {y_true[worst]:.2f}, predicted {y_pred[worst]:.2f}, "
      f"off by {abs(y_true[worst] - y_pred[worst]):.2f}")

`np.abs(y_true - y_pred).mean()` is the whole metric: subtract elementwise, drop the signs,
average. This is exactly the formula from 00-01, now written on arrays instead of in a loop - and
it is worth noticing that `mean_absolute_error` from scikit-learn does precisely this.

`np.argmax` returns the **position** of the largest value, not the value. That distinction is the
useful one: the position lets you go back to the row and ask *what was different about it* - which
is the first move of error analysis (07-05).

For the top ten instead of the top one, `np.argsort(np.abs(errors))[-10:]` gives the positions,
and this pairs with 01-02's `worst_n` on records: get the identity, not just the number.

## E8 · Why standardising changed nothing here

Ordinary linear regression fits `y = b0 + b1*x1 + b2*x2 + ...` with no constraint on the
coefficients. Rescale a feature and the fitted coefficient rescales by exactly the inverse, giving
identical predictions. The model is **scale-invariant**, so standardisation is cosmetic - it
changes what the coefficients mean, not what the model does.

**Two families where it matters enormously:**

- **Regularised models** (ridge, lasso - 05-09) penalise the size of the coefficients. A feature
  measured in thousands gets a tiny coefficient, so the penalty barely touches it, while a feature
  measured in units gets a large one and is penalised hard. Without standardisation the penalty
  falls arbitrarily according to the units someone chose. The same applies to neural networks
  (module 10), where unscaled inputs make training unstable.
- **Distance-based models** (k-nearest neighbours - 06-10; k-means - 08-03) compute distances
  across features. A count in the thousands overwhelms a temperature in the tens, so the distance -
  and therefore every neighbour and every cluster - is decided by whichever column happens to have
  the biggest numbers. You saw this in 00-03: clustering on different columns gave different
  "discoveries".

**The transferable lesson:** a preprocessing step is not automatically good. It must be justified
by the model you are about to fit. 04-06 makes that justification part of the workflow.

## E9 · The student averages

In [ ]:
scores = np.array([[70, 80, 90, 100], [60, 65, 70, 75], [88, 92, 96, 100]])   # 3 students, 4 tests

averages = scores.mean(axis=0)
print("scores.shape  ", scores.shape, " -> 3 students, 4 tests")
print("averages      ", averages, " shape", averages.shape, " <- FOUR numbers for THREE students")
print("above         \n", scores - averages)

correct = scores.mean(axis=1, keepdims=True)
print("\ncorrect per-student averages:\n", correct, " shape", correct.shape)
print("correct 'above own average':\n", scores - correct)

**`averages` contains the average of each *test* across students** - four numbers - not the
average of each student. The axis rule says it immediately: `axis=0` removes the 3, which is the
student dimension.

**`above` then broadcasts `(3, 4) - (4,)`, which succeeds**, and gives "how far each score is above
that test's average across students". A real quantity, plausibly distributed, and not the one that
was asked for.

**The `.shape` print is the whole diagnosis.** Four numbers where there are three students is
impossible, and it takes one second to see. This is why the instinct is to print shapes rather
than to inspect values - values look plausible, shapes do not lie.

**The fix** is `axis=1`, and `keepdims=True` so the result has shape `(3, 1)` and broadcasts down
the rows instead of across the columns. Without `keepdims` you would get `(3,)`, and `(3, 4) -
(3,)` raises - which would at least have been loud.

## E10 · "Why is NumPy faster?"

> Three reasons, and the language it is written in is only one of them. First, the data is one
> type in one contiguous block, so the processor can walk it without chasing pointers or checking
> a type on every element - a Python list is an array of pointers to boxed objects scattered in
> memory. Second, the loop itself runs once in compiled code instead of once per element in the
> interpreter, which removes the per-iteration overhead that dominates small operations. Third,
> the compiled loop can use the processor's vector instructions to handle several elements per
> cycle, and can hand large operations to optimised linear-algebra libraries. The consequence
> worth stating is that the win comes from *not returning to Python per element*, which is why
> `arr.sum()` is fast and `sum(arr)` is not.

**What the interviewer is checking:** whether you understand that the cost is the interpreter
round-trip per element. That understanding is what tells you which loops are worth vectorising -
the inner one over a million rows, not the outer one over five experiments.

## E11 · Drop the NaN rows, then standardise

In [ ]:
big = rng.normal(0, 1, (1000, 5))
big[np.array([3, 17, 900]), 2] = np.nan          # plant a few missing values

keep = ~np.isnan(big).any(axis=1)                # 1. which rows are complete?
clean = big[keep]                                # 2. keep only those
scaled = (clean - clean.mean(axis=0)) / clean.std(axis=0)   # 3. now standardise

print("rows in:", len(big), " rows kept:", int(keep.sum()))
print("column means after scaling:", scaled.mean(axis=0).round(9))
print("what standardising FIRST would have given:", (big - big.mean(axis=0)).mean(axis=0).round(3))

**Why the order is forced:** `mean` over a column containing `NaN` returns `NaN`. Standardise
first and every value in that column becomes `NaN`, so the "drop rows with NaN" step then deletes
the entire dataset. The last line of the output shows the poisoning: one missing value makes a
whole column mean `NaN`.

Line 1 deserves reading slowly: `np.isnan(big)` gives a `(1000, 5)` boolean array; `.any(axis=1)`
collapses the 5, giving one boolean per **row** - "does this row contain any NaN?"; and `~`
negates it to "is this row complete?". That is `axis=1` used correctly, and it is the exception
that proves the rule from the chapter: `axis=1` is right here because the question genuinely is
about rows.

**What a careful person adds:** print how many rows were dropped, every time. Losing 3 of 1,000 is
routine; losing 400 means the missingness is systematic and dropping them is throwing away a
pattern, not cleaning noise. Chapter 02-04 is about telling those apart, and 04-06 covers
imputing instead of dropping.

## E12 · An image, `(256, 256, 3)`

| Expression | Shape | What it is |
|---|---|---|
| (a) `img.mean(axis=(0, 1))` | `(3,)` | Height and width both disappear: **the average colour** - one value per channel |
| (b) `img.mean(axis=2)` | `(256, 256)` | The channel axis disappears: **a greyscale image** - one brightness per pixel |
| (c) `img.max()` | scalar | The single brightest value anywhere - useful mostly for checking whether pixels are 0-255 or 0-1 |

**Greyscale: (b). Average colour: (a).**

Two notes worth having before module 11:

- Passing a *tuple* of axes collapses several at once. The rule generalises unchanged: every axis
  you name disappears.
- A true greyscale conversion weights the channels (roughly 0.21 red, 0.72 green, 0.07 blue)
  because human eyes are far more sensitive to green. A plain mean is the *arithmetic* answer and
  a slightly wrong *perceptual* one - a good small example of a computation being correct and the
  wrong thing.

## E13 · Explaining standardisation

> Different measurements come in different units - degrees, percentages, counts in the thousands -
> so their numbers are not comparable. Standardising rewrites each one as "how far above or below
> its own average this is, in typical steps", so a value of 2 means the same thing in every column.
> It is like converting several currencies into one before adding them up.

(64 words.)

**Where the currency comparison stops being accurate:** exchange rates convert to a shared unit
that still has meaning - euros are a thing. A standardised value has no unit at all; it is a
position within a distribution. And it is a position relative to *the data you standardised on*,
so the same physical temperature gets a different standardised value in a different dataset.

## E14 · Pairwise distances, and when a clever trick is wrong

In [ ]:
def pairwise_distances(A, B):
    """Euclidean distance between every row of A (n, d) and every row of B (m, d) -> (n, m)."""
    diff = A[:, None, :] - B[None, :, :]          # (n, 1, d) - (1, m, d) -> (n, m, d)
    return np.sqrt((diff ** 2).sum(axis=2))       # collapse d -> (n, m)


A = rng.normal(0, 1, (4, 3))
B = rng.normal(0, 1, (5, 3))
fast = pairwise_distances(A, B)

slow = np.zeros((4, 5))
for i in range(4):
    for j in range(5):
        slow[i, j] = np.sqrt(((A[i] - B[j]) ** 2).sum())

print("shape", fast.shape, " matches the double loop:", np.allclose(fast, slow))

In [ ]:
n = m = 50_000
d = 20
gb = n * m * d * 8 / 1e9
print(f"the intermediate (n, m, d) array would need {gb:,.0f} GB")
print(f"the answer itself, (n, m), needs only {n * m * 8 / 1e9:.1f} GB - still too much")

The broadcasting version is elegant, correct, and **400 GB** of intermediate array at 50,000
rows - on a machine that has 16.

The trick works by materialising every pairwise difference at once - an `(n, m, d)` array - even
though only the summed distance is wanted. At small sizes that is free. At scale it is
catastrophic, and the failure is a crash or a machine that starts swapping, not a wrong number.

**What to do instead:**

- `sklearn.metrics.pairwise_distances`, which uses the algebraic identity
  `|a-b|² = |a|² + |b|² - 2a·b` to get the answer from a matrix multiplication with no
  three-dimensional intermediate.
- Process in chunks of a few thousand rows if you need the full matrix.
- Ask whether you need it at all. You usually want the *k nearest*, not every distance, and a
  spatial index (`sklearn.neighbors.BallTree`) answers that without computing them all - which is
  what makes k-nearest neighbours usable in 06-10.

**The transferable lesson:** *"it broadcasts"* is not the same as *"it is a good idea"*. When a
vectorised expression creates an intermediate array with more dimensions than either input,
compute its size before running it. Elegance that allocates a dimension you do not need is the
most common way a working notebook dies on the real dataset.

---

## Where to go next

Back to the chapter for the mastery check and flashcards, then **01-04 · pandas I**, where these
arrays acquire column names, mixed types and an index.